# 03 — OLAP Analysis
Roll-up, Drill-down, Slice, and Dice operations.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

df = pd.read_csv('../data/processed/apsrtc_features.csv')
df['date'] = pd.to_datetime(df['date'], errors='coerce')
print('Shape:', df.shape)

In [ ]:
# ROLL-UP: monthly aggregation
print('=== ROLL-UP: Monthly Demand ===')
monthly = df.groupby(['year','month_num'])['passengers'].agg(['sum','mean','count'])
monthly.columns = ['total_passengers','avg_passengers','trips']
print(monthly.to_string())

In [ ]:
# DRILL-DOWN: 2024 by month
print('=== DRILL-DOWN: 2024 Month-by-Month ===')
y2024 = df[df['year'] == 2024].groupby('month_num')['passengers'].sum()
y2024.plot(kind='bar', title='2024 Monthly Total Passengers', figsize=(10, 4))
plt.tight_layout()
plt.show()

In [ ]:
# SLICE: Volvo AC only
print('=== SLICE: Volvo AC ===')
volvo = df[df['bus_type'].str.contains('Volvo', case=False, na=False)]
print(f'Volvo trips: {len(volvo)}')
print(volvo.groupby('route')['passengers'].mean().sort_values(ascending=False))

In [ ]:
# DICE: August + Weekday
print('=== DICE: August Weekdays ===')
aug_wkd = df[(df['month_num'] == 8) & (df['is_weekend'] == 0)]
result = aug_wkd.groupby('route')['passengers'].agg(['mean','count'])
result.columns = ['avg_passengers','trips']
print(result.sort_values('avg_passengers', ascending=False))

In [ ]:
# KPI summary
kpis = {
    'Total Routes':    df['route'].nunique(),
    'Total Trips':     len(df),
    'Total Passengers': int(df['passengers'].sum()),
    'Avg Demand/Trip':  round(df['passengers'].mean(), 1),
    'Peak Demand':     int(df['passengers'].max()),
    'Min Demand':      int(df['passengers'].min()),
    'Avg Revenue/Trip': round(df['revenue'].mean(), 0),
}
for k, v in kpis.items():
    print(f'{k:25s}: {v}')